In [ ]:
import os
import random
import numpy as np
import scipy.io as spio

def phi(inVolt,threshold):
    currArr = inVolt/threshold
    return np.where(currArr>1,1,currArr)

def galvesLocherbachSimulator(unitCount,connGraph,remain,thresh,timeSteps):
    random.seed(2022) #Pick a RNG seed, does not matter which one, just make sure it consistent across all runs
    outputSimmedSpikeTrain = 0
    sampV = np.random.randint(0,thresh,size=(unitCount,))
    for t in range(timeSteps):
        spikeProb = phi(sampV,thresh)
        randArr = np.random.uniform(0,thresh*3,size=(unitCount,))
        trueSpike = np.reshape(np.where(spikeProb>randArr,1,0),(len(spikeProb),1)).astype(int)
        if np.shape(outputSimmedSpikeTrain) == ():
            outputSimmedSpikeTrain = trueSpike
        else:
            outputSimmedSpikeTrain = np.concatenate((outputSimmedSpikeTrain,trueSpike),axis=1)
        voltSpike = np.where(spikeProb>randArr,0,spikeProb)
        voltSpike = voltSpike*remain
        sampV = voltSpike+np.sum(voltSpike*connGraph)
    return outputSimmedSpikeTrain.astype(int)

def splitTrainIntoTrials(inputSimmedTrain,inputInds):
    outputTrialTrains = [None]
    for i in range(1,len(inputInds)-1,2):
        currTrain = inputSimmedTrain[:,inputInds[i]:inputInds[int(i+1)]]
        outputTrialTrains.append(currTrain)
    return outputTrialTrains[1:]

def loadValuesFromFile(fileOfInterest):
    rmat = spio.loadmat(fileOfInterest)
    regionIndex = rmat.get('regionIndex').flatten()
    graphOfConns = rmat.get('connectionAll')
    fields = rmat.get('unitFields')
    modelValues = rmat.get('modelValuesAll').flatten() #threshold, remain
    timingInds = rmat.get('timingInds').flatten()
    simmedTrain = galvesLocherbachSimulator(len(regionIndex),graphOfConns,modelValues[1],modelValues[0],timingInds[11])
    simmedTrainTrialSplit = splitTrainIntoTrials(simmedTrain,timingInds[:11])
    outputDict = {'regionIndex':regionIndex,'unitFields':fields}
    for a,trialTrain in enumerate(simmedTrainTrialSplit):
        outputDict.update({f'Trial0{int(a+1)}':trialTrain})
    return outputDict

def generateTrueTrains(allRawPaths,allSavPaths):
    for z,singleRawP in enumerate(allRawPaths):
        singleSavP = allSavPaths[z]
        for aRoot,aDirs,aFiles in os.walk(singleRawP):
            for aFile in aFiles:
                if aFile.endswith('.mat'):
                    currRawFile = os.path.join(singleRawP,aFile)
                    currSavFile = os.path.join(singleSavP,aFile)
                    dictToSave = loadValuesFromFile(currRawFile)
                    var = spio.savemat(currSavFile,dictToSave)
                    print(aFile)
        print(f'Built from {singleRawP}')
    return 'Done'

rawPs = []

savPs = []

trainSimmed = generateTrueTrains(rawPs,savPs)
print(trainSimmed)
